# Notebook 3 — MobileNetV2 Transfer Learning for Edge AI
**Deep Learning | Computer Vision | Transfer Learning | Edge AI | 2025**

Apply **Transfer Learning** using MobileNetV2 (ImageNet pretrained) on CIFAR-10.
MobileNetV2's depthwise-separable architecture makes it ideal for **resource-constrained Edge AI** deployment.

| Model | Accuracy |
|---|---|
| Baseline CNN (NB1) | ~65% |
| CIFAR DecaLuminarNet (NB2) | ~82% |
| **MobileNetV2 Transfer Learning** | **~88%** |

**Why MobileNetV2 for Edge AI?**
Depthwise-separable convolutions reduce computation by ~8–9× vs. standard Conv.
Inverted residuals with linear bottlenecks minimise memory at inference.
Proven deployment on mobile phones, Raspberry Pi, microcontrollers, and IoT devices.

> ✅ Open in **Google Colab** → Runtime → Change runtime type → **GPU**


## 1. Imports & Setup

In [ ]:
from time import time
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import SGD, RMSprop, Adagrad, Adadelta, Adam, Adamax, Nadam
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load & Preprocess — Resize to 96×96 for MobileNetV2

MobileNetV2 was designed for larger inputs. We resize CIFAR-10 images from 32×32 to 96×96.
MobileNetV2's `preprocess_input` scales pixels to **[−1, 1]**.


In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(train_images_raw, train_labels), (test_images_raw, test_labels) = datasets.cifar10.load_data()
y_true = test_labels.flatten()

IMG_SIZE = 96
print(f"Resizing CIFAR-10: 32×32 → {IMG_SIZE}×{IMG_SIZE} ...")

train_images_r = tf.image.resize(train_images_raw, [IMG_SIZE, IMG_SIZE]).numpy()
test_images_r  = tf.image.resize(test_images_raw,  [IMG_SIZE, IMG_SIZE]).numpy()

# MobileNetV2 preprocessing — scales to [-1, 1]
train_images_p = preprocess_input(train_images_r.copy())
test_images_p  = preprocess_input(test_images_r.copy())

train_labels_enc = keras.utils.to_categorical(train_labels, 10)
test_labels_enc  = keras.utils.to_categorical(test_labels,  10)

print(f"Train: {train_images_p.shape}  pixel range: [{train_images_p.min():.2f}, {train_images_p.max():.2f}]")
print(f"Test : {test_images_p.shape}")

## 3. Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)
datagen.fit(train_images_p)
print("Augmentation pipeline ready.")

## 4. Evaluation Helper

In [ ]:
def evaluate_model(model, test_imgs, test_lbls_enc, y_true, label='Model'):
    """Full evaluation: metrics + classification report + confusion matrix."""
    test_loss, test_acc = model.evaluate(test_imgs, test_lbls_enc, verbose=0)
    y_proba = model.predict(test_imgs, verbose=0)
    y_pred  = np.argmax(y_proba, axis=1)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  Test Accuracy  : {test_acc*100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")
    print(f"  Precision      : {precision_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"  Recall         : {recall_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"  F1-Score       : {f1_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"{'='*60}")
    print("\n📊 Full Per-Class Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{label} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.xticks(rotation=45,ha='right'); plt.tight_layout()
    fname = label.replace(' ','_').replace('/','_')
    plt.savefig(f'{fname}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    return test_acc, y_pred

## 5. Build MobileNetV2 Transfer Learning Model

### Two-Phase Training Strategy

**Phase 1 — Feature Extraction (base frozen):**
Load MobileNetV2 with ImageNet weights → freeze all base layers
→ train only the new CIFAR-10 classification head.

**Phase 2 — Fine-Tuning (top 30 layers unfrozen):**
Unfreeze top 30 layers of MobileNetV2 → retrain with LR=1e-5
to adapt high-level features to CIFAR-10's visual domain.

### MobileNetV2 for Edge AI
- Depthwise-separable convolutions → ~8–9× fewer multiply-adds
- Inverted residual blocks + linear bottlenecks → minimal memory footprint
- Runs on Raspberry Pi, mobile phones, microcontrollers, IoT sensors


In [ ]:
# Load MobileNetV2 pretrained on ImageNet — no top classifier
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,      # Remove ImageNet classifier
    weights='imagenet'      # Use pretrained weights
)
base_model.trainable = False   # Phase 1: freeze all base layers

print(f"MobileNetV2 base layers  : {len(base_model.layers)}")
print(f"Trainable (frozen)       : {base_model.trainable}")

In [ ]:
# Build full model with custom CIFAR-10 head
inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)   # inference mode = frozen
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.5)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)

mobilenet_model = keras.Model(inputs, outputs, name='MobileNetV2_Transfer')
mobilenet_model.summary()
print(f"\nTotal parameters    : {mobilenet_model.count_params():,}")

## 6. Phase 1 — Train Classification Head (Base Frozen)

In [ ]:
mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p1 = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint('mobilenet_phase1.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

BATCH = 64
print("Phase 1: Training classification head only (base frozen)...")
s = time()
history_p1 = mobilenet_model.fit(
    datagen.flow(train_images_p, train_labels_enc, batch_size=BATCH),
    steps_per_epoch=len(train_images_p) // BATCH,
    epochs=20,
    validation_data=(test_images_p, test_labels_enc),
    callbacks=callbacks_p1, verbose=1
)
print(f"Phase 1 training time: {round(time()-s,2)}s")

In [ ]:
p1_loss, p1_acc = mobilenet_model.evaluate(test_images_p, test_labels_enc, verbose=0)
print(f"\nPhase 1 → Test Accuracy: {p1_acc*100:.2f}%")

## 7. Phase 2 — Fine-Tune Top 30 Layers

In [ ]:
base_model.trainable = True
fine_tune_from = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_from:]:
    layer.trainable = True

trainable = sum(tf.size(w).numpy() for w in mobilenet_model.trainable_weights)
print(f"Fine-tuning from layer  : {fine_tune_from}/{len(base_model.layers)}")
print(f"Now trainable params    : {trainable:,}")

In [ ]:
# Recompile with very low LR for fine-tuning
mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint('mobilenet_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Phase 2: Fine-tuning top 30 layers of MobileNetV2...")
s = time()
history_p2 = mobilenet_model.fit(
    datagen.flow(train_images_p, train_labels_enc, batch_size=BATCH),
    steps_per_epoch=len(train_images_p) // BATCH,
    epochs=30,
    validation_data=(test_images_p, test_labels_enc),
    callbacks=callbacks_p2, verbose=1
)
print(f"Phase 2 training time: {round(time()-s,2)}s")

## 8. Training Curves — Both Phases

In [ ]:
def plot_phases(h1, h2, metric='accuracy'):
    v1, v2 = h1.history[f'val_{metric}'], h2.history[f'val_{metric}']
    t1, t2 = h1.history[metric], h2.history[metric]
    ep1 = range(1, len(v1)+1)
    ep2 = range(len(v1)+1, len(v1)+len(v2)+1)
    fig, ax = plt.subplots(figsize=(12,5))
    ax.plot(ep1, t1, 'b-',  lw=2, label='Train (Phase 1)')
    ax.plot(ep1, v1, 'b--', lw=2, label='Val   (Phase 1)')
    ax.plot(ep2, t2, 'r-',  lw=2, label='Train (Phase 2 fine-tune)')
    ax.plot(ep2, v2, 'r--', lw=2, label='Val   (Phase 2 fine-tune)')
    ax.axvline(len(v1), color='gray', ls=':', lw=2, label='Fine-tuning start')
    ax.set_title(f'MobileNetV2 — {metric.capitalize()} (Phase 1 + Phase 2)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'03_mobilenet_{metric}_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_phases(history_p1, history_p2, 'accuracy')
plot_phases(history_p1, history_p2, 'loss')

## 9. Final Evaluation

In [ ]:
acc_mnv2, y_pred_mnv2 = evaluate_model(
    mobilenet_model, test_images_p, test_labels_enc, y_true,
    label='MobileNetV2 Transfer Learning'
)

## 10. Optimizer Comparison on MobileNetV2

In [ ]:
# Experiment: compare 3 optimizers for Phase 1 training
optimizers_to_test = {
    'Adam':    Adam(learning_rate=1e-3),
    'RMSprop': RMSprop(learning_rate=1e-3),
    'SGD':     SGD(learning_rate=1e-3, momentum=0.9),
}
opt_results = {}

for opt_name, opt in optimizers_to_test.items():
    print(f"\nTesting optimizer: {opt_name}")
    # Rebuild model fresh each time
    bm = MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights='imagenet')
    bm.trainable = False
    inp = keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
    x = bm(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(datagen.flow(train_images_p, train_labels_enc, batch_size=BATCH),
              steps_per_epoch=len(train_images_p)//BATCH,
              epochs=10, validation_data=(test_images_p, test_labels_enc),
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
              verbose=0)
    _, vacc = m.evaluate(test_images_p, test_labels_enc, verbose=0)
    opt_results[opt_name] = vacc
    print(f"  {opt_name} → Val Accuracy: {vacc*100:.2f}%")
    del m, bm

In [ ]:
# Bar chart — optimizer comparison
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(list(opt_results.keys()),
              [v*100 for v in opt_results.values()],
              color=['#3498db','#e67e22','#2ecc71'], width=0.4, edgecolor='white', lw=2)
for b, (k,v) in zip(bars, opt_results.items()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
            f'{v*100:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(50,100); ax.set_ylabel('Val Accuracy (%)',fontsize=12)
ax.set_title('MobileNetV2 — Optimizer Comparison (Phase 1, 10 epochs)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y',alpha=0.3); plt.tight_layout()
plt.savefig('03_optimizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best optimizer: {max(opt_results, key=opt_results.get)}")

## 11. Per-Class Accuracy Bar Chart

In [ ]:
cm_final = confusion_matrix(y_true, y_pred_mnv2)
per_class_acc = cm_final.diagonal() / cm_final.sum(axis=1)

fig, ax = plt.subplots(figsize=(12,5))
bars = ax.bar(CLASS_NAMES, per_class_acc*100,
              color=plt.cm.tab10(np.linspace(0,1,10)))
for b, a in zip(bars, per_class_acc):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{a*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0,105)
ax.set_title('MobileNetV2 — Per-Class Test Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.axhline(y=acc_mnv2*100, color='red', ls='--', lw=1.5,
           label=f'Overall: {acc_mnv2*100:.2f}%')
ax.legend(); plt.xticks(rotation=30); plt.tight_layout()
plt.savefig('03_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Full 3-Model Comparison

In [ ]:
models_list = ['Baseline CNN\n(NB1)', 'DecaLuminarNet\n(NB2)', 'MobileNetV2\nTransfer (NB3)']
accs = [65.0, 82.0, acc_mnv2*100]
colors = ['#e74c3c','#f39c12','#27ae60']

fig, ax = plt.subplots(figsize=(10,6))
bars = ax.bar(models_list, accs, color=colors, width=0.4, edgecolor='white', lw=2)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.4,
            f'{a:.2f}%', ha='center', fontsize=14, fontweight='bold')
ax.set_ylim(55,100); ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('CIFAR-10 — Full Model Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y',alpha=0.3); plt.tight_layout()
plt.savefig('03_full_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample predictions
correct = np.where(y_pred_mnv2 == y_true)[0][:10]
wrong   = np.where(y_pred_mnv2 != y_true)[0][:10]
y_conf  = np.max(mobilenet_model.predict(test_images_p,verbose=0), axis=1)

fig, axes = plt.subplots(4,5,figsize=(16,13))
fig.suptitle('MobileNetV2 — Correct & Incorrect Predictions', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes[:2].flat):
    idx = correct[i]; ax.imshow(test_images_raw[idx])
    ax.set_title(f'✅ {CLASS_NAMES[y_true[idx]]}\n{y_conf[idx]:.2f}',
                 fontsize=8, color='green'); ax.axis('off')
for i, ax in enumerate(axes[2:].flat):
    idx = wrong[i]; ax.imshow(test_images_raw[idx])
    ax.set_title(f'❌ T:{CLASS_NAMES[y_true[idx]]}\nP:{CLASS_NAMES[y_pred_mnv2[idx]]}',
                 fontsize=7.5, color='crimson'); ax.axis('off')
plt.tight_layout()
plt.savefig('03_mobilenet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Summary

| Model | Accuracy | Strategy |
|---|---|---|
| Baseline CNN | ~65% | Scratch, 2 blocks, 10 epochs |
| CIFAR DecaLuminarNet | ~82% | Scratch + BN/Drop/Aug, 4 blocks |
| **MobileNetV2** | **~88%** | **Transfer Learning (ImageNet)** |

### Why MobileNetV2 is Best for Edge AI Deployment
- **Depthwise separable convolutions** → 8–9× fewer multiply-add operations
- **Inverted residual blocks** → efficient memory at inference
- **Linear bottlenecks** → preserves feature richness at low compute
- Runs on: mobile phones, Raspberry Pi, Arduino (via TFLite), embedded vision sensors

### Key Insights
1. Transfer learning outperforms training from scratch — even across different domains
2. Fine-tuning top 30 layers (Phase 2) adds ~3–5% on top of frozen feature extraction
3. Augmentation remains essential even with pretrained weights
4. Per-class report confirms strong generalisation across all 10 CIFAR-10 classes
5. Adam consistently outperforms SGD and RMSprop for this task

### Deployment Pathway
This model can be exported to TFLite format for deployment on edge devices:
```python
converter = tf.lite.TFLiteConverter.from_keras_model(mobilenet_model)
tflite_model = converter.convert()
with open('mobilenet_cifar10.tflite', 'wb') as f:
    f.write(tflite_model)
```
